# Saved Queries

A **Query** is a named, reusable set of image filters. You save the filters once and reference the query by its slug wherever you list or count images.

Queries are dynamic. The filters are stored as-is and compiled to a search on demand, so a query always reflects the current data. Nothing is precompiled or frozen at save time.

The `filters` you save are the same parameters the images endpoint accepts (for example `sources`, `tags`, `aspect_ratio__gt`). Saving a query is just freezing the current image search under a name.

What this notebook covers:

| | |
|---|---|
| [Initialize the client](#Initialize-the-client) | connect |
| [Create a query](#Create-a-query) | save a filter set under a slug |
| [List queries](#List-queries) | every saved query |
| [Get one query](#Get-one-query) | fetch a single query by slug |
| [Use a query to filter images](#Use-a-query-to-filter-images) | `images.list(query=...)` |
| [Update a query](#Update-a-query) | change name, description or filters |
| [Delete a query](#Delete-a-query) | remove it |

## Initialize the client

In [1]:
import os
from dataroom_client import DataRoomClient

os.environ["DATAROOM_API_KEY"] = 'YOUR_KEY_HERE'
os.environ["DATAROOM_API_URL"] = 'http://localhost:8000/api/'

DataRoom = DataRoomClient()

## Create a query

Pass a `slug` (used in the URL and to reference the query), a display `name`, and the `filters`. The filters are validated by compiling them once, so an invalid filter (for example an unknown tag) is rejected at create time.

In [2]:
from dataroom_client import DataRoomError

try:
    await DataRoom.queries.delete('square-or-wider')  # make reruns idempotent
except DataRoomError:
    pass

query = await DataRoom.queries.create(
    slug='square-or-wider',
    name='Square or wider images',
    filters={'aspect_ratio__gte': 1.0},
    description='Images at least as wide as they are tall.',
)
query

{'slug': 'square-or-wider',
 'name': 'Square or wider images',
 'description': 'Images at least as wide as they are tall.'}

## List queries

In [3]:
queries = await DataRoom.queries.list()
[(q['slug'], q['name']) for q in queries]

[('square-or-wider', 'Square or wider images')]

## Get one query

The detail response returns the stored `filters` along with the name, description, and author. Use `count_images(query=...)` (shown below) when you want a live image count.

In [4]:
query = await DataRoom.queries.get('square-or-wider')
query['filters']

{'aspect_ratio__gte': 1.0}

## Use a query to filter images

Pass `query=<slug>` to any image endpoint. The saved filters are applied on top of (and combined with) any other filters you pass.

In [5]:
images = await DataRoom.images.list(query='square-or-wider', fields=['id'])
count = await DataRoom.images.count(query='square-or-wider')
print('returned:', len(images), 'total:', count)

returned: 1000 total: 103008


## Update a query

Change the name, description, or filters. The next list or count reflects the new filters immediately.

In [6]:
await DataRoom.queries.update(
    slug='square-or-wider',
    filters={'aspect_ratio__gte': 1.0, 'short_edge__gte': 64},
)
query = await DataRoom.queries.get('square-or-wider')
query['filters']

{'short_edge__gte': 64, 'aspect_ratio__gte': 1.0}

## Delete a query

Removes the saved query. Images are not affected - a query is only a stored set of filters.

In [7]:
await DataRoom.queries.delete('square-or-wider')